This notebook contains the code we used to run scVI for 10 random seeds on the tabula dataset: it is very expensive to run!

We reccomend changing the paths and saving the highly variable genes object to a different path, since that is expensive to calculate.

In [1]:
import scanpy as sc
import scvi
import numpy as np
from tqdm import tqdm
import torch
import random

In [2]:
full_tabula_ad = sc.read("/lfs/local/0/yanay/new_tabula_sapiens.h5ad") # Change this path to export_data for the h5ad!
full_tabula_ad

/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 1194952 × 61852
    obs: 'donor', 'tissue', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'old_index', 'assay', 'sample_id', 'sample', 'replicate', '10X_run', '10X_barcode', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'age', 'sex', 'ethnicity'
    var: 'ensembl_id', 'gene_symbol', 'genome', 'mt', 'ercc', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'

# HVG

In [ ]:
#adata = full_tabula_ad.copy()

#sc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3", subset=True)
#print(f"Size: {adata.X.shape}")

In [10]:
#adata.write("/lfs/local/0/yanay/tabula3k.h5ad") # change this path because it is expensive to calcualte HVG, so you should save the file

In [11]:
adata = sc.read("/lfs/local/0/yanay/tabula3k.h5ad") # change this path because it is expensive to calcualte HVG, so you should save the file

/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


## Run scVI

In [28]:
for seed in np.arange(10):
    seed = int(seed)
    print(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    scvi.model.SCVI.setup_anndata(
        adata,
        categorical_covariate_keys=["sample_id"],
    )
    scvi.settings.batch_size=256
    scvi.settings.drop_last=True
    
    model = scvi.model.SCVI(adata,)
    model.train(batch_size=132)
    
    
    latent = model.get_latent_representation()
    stacked_scvi = sc.AnnData(latent)
    stacked_scvi.obs = adata.obs
    stacked_scvi.write(location + f"/full_3k_seed_{seed}.h5ad")

0


/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_arraylike_field.py:407: UserWarning: Category 126 in adata.obs['sample_id'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(df, key, key, categorical_dtype=categorical_dtype)
Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be 

Epoch 7/7: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [07:02<00:00, 61.13s/it, v_num=1, train_loss_step=193, train_loss_epoch=207]

`Trainer.fit` stopped: `max_epochs=7` reached.


Epoch 7/7: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [07:02<00:00, 60.32s/it, v_num=1, train_loss_step=193, train_loss_epoch=207]
1


...

Epoch 7/7: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [07:23<00:00, 63.35s/it, v_num=1, train_loss_step=193, train_loss_epoch=207]
9


/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_arraylike_field.py:407: UserWarning: Category 126 in adata.obs['sample_id'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(df, key, key, categorical_dtype=categorical_dtype)
Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be 

Epoch 7/7: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [07:20<00:00, 62.90s/it, v_num=1, train_loss_step=216, train_loss_epoch=206]

`Trainer.fit` stopped: `max_epochs=7` reached.


Epoch 7/7: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [07:20<00:00, 62.97s/it, v_num=1, train_loss_step=216, train_loss_epoch=206]
